In [528]:
import pandas as pd
import numpy as np
import os 

In [529]:
#razlikovace se jer je za xgb label encoded, a za dnn one hot
file = 'features_for_XGBoost.csv'
if os.path.exists(file):
    sve = pd.read_csv(file)
else:
    sve = pd.DataFrame()

In [530]:
trke = pd.read_csv('all_races.csv')

In [531]:
trke.columns


Index(['season', 'round', 'race_name', 'date', 'time', 'circuit', 'country'], dtype='object')

In [532]:
trke = trke.drop(['date', 'time', 'country'], axis=1)

In [533]:
staze = set(zip(trke['circuit']))

In [534]:
staze = pd.DataFrame(staze)

In [535]:
staze.to_csv('non_processed_circuits.csv', index=False)
print(f"Sacuvano")

Sacuvano


In [536]:
staze = pd.read_csv('processed_circuits.csv')

In [537]:
trke = trke.merge(staze, on='circuit', how='left')    
    

In [538]:
vreme = pd.read_csv('all_weather.csv')
vreme = vreme.drop(['date', 'time', 'race_name'], axis=1)

In [539]:
#primenjujemo binning za kisu
bins = [ -1, 0, 1, 5, 100 ]
labels = [0, 1, 2, 3]
vreme['rain_bin'] = pd.cut(vreme['Rainfall'], bins=bins, labels=labels)


In [540]:
#binning za jedan sample vetra
def wind_dir_bin(deg):
    if (deg >= 337.5) or (deg < 22.5):
        return 0  # N
    elif deg < 67.5:
        return 1  # NE
    elif deg < 112.5:
        return 2  # E
    elif deg < 157.5:
        return 3  # SE
    elif deg < 202.5:
        return 4  # S
    elif deg < 247.5:
        return 5  # SW
    elif deg < 292.5:
        return 6  # W
    else:
        return 7  # NW

In [541]:
vreme['wind_direction'] = vreme['WindDirection'].apply(wind_dir_bin)

In [542]:
vreme = vreme.drop(['Rainfall', 'WindDirection'], axis=1)

In [543]:
trke = trke.merge(vreme, on=['season', 'round'], how='left')    
    

In [544]:
trke['is_sprint_weekend'] = False

In [545]:
#u formatu [season, round]
sprint_vikendi = [
    [2021, 10], 
    [2021, 14], 
    [2021, 19], 

    [2022, 4], 
    [2022, 11], 
    [2022, 21], 

    [2023, 4], 
    [2023, 10], 
    [2023, 13], 
    [2023, 18], 
    [2023, 19], 
    [2023, 21], 

    [2024, 5],
    [2024, 6], 
    [2024, 11], 
    [2024, 19], 
    [2024, 21], 
    [2024, 23] 
]

In [546]:
for i in range(len(trke)):
    if [trke.loc[i]['season'], trke.loc[i]['round']] in sprint_vikendi:
        trke.at[i, 'is_sprint_weekend'] = True

In [547]:
#FILTRIRAN DATASET sa neta
sa_neta = pd.read_csv('cleaned_dataset.csv')

In [548]:
#svaki red se mnozi sa 20 jer su prethodni podaci zajednicki
# n = 20

# trke = trke.loc[np.repeat(trke.index.values, n)].reset_index(drop=True)

In [549]:
trke

,season,round,race_name,circuit,lap_length,number_of_laps,number_of_corners,pit_lane_time_loss,average_overtakes_per_circuit,average_pit_stops_per_race,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend
0,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,5.303,58.0,14.0,NaN,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
1,2018,2,Bahrain Grand Prix,Bahrain International Circuit,5.412,57.0,15.0,NaN,NaN,NaN,27.982524,47.363107,1009.494175,32.198058,0.958252,0,4,False
2,2018,3,Chinese Grand Prix,Shanghai International Circuit,5.451,56.0,16.0,NaN,NaN,NaN,19.446429,24.089286,1018.131250,37.019643,1.837500,1,3,False
3,2018,4,Azerbaijan Grand Prix,Baku City Circuit,6.003,51.0,20.0,,NaN,NaN,16.661404,45.651754,1021.913158,25.251754,2.222807,0,3,False
4,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,4.655,66.0,14.0,NaN,NaN,NaN,16.050476,52.286667,1001.541905,32.339048,1.952381,1,2,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,2024,20,Mexico City Grand Prix,Autódromo Hermanos Rodríguez,4.304,71.0,17.0,NaN,NaN,NaN,19.888679,51.050314,785.252830,35.602516,2.150943,0,6,False
145,2024,21,São Paulo Grand Prix,Autódromo José Carlos Pace,4.309,71.0,15.0,NaN,NaN,NaN,21.662687,85.258706,926.456716,25.769652,0.735821,1,4,True
146,2024,22,Las Vegas Grand Prix,Las Vegas Strip Street Circuit,6.201,50.0,17.0,NaN,NaN,NaN,17.825175,47.860140,935.892308,17.723776,2.898601,0,5,False
147,2024,23,Qatar Grand Prix,Losail International Circuit,5.419,57.0,16.0,NaN,NaN,NaN,18.926490,57.251656,1015.500000,22.773510,1.657616,0,2,True


In [550]:
kvalifikacije = pd.read_csv('all_qualies.csv')
kvalifikacije = kvalifikacije.drop(['race_name','Q1_time', 'Q2_time', 
                                    'Q3_time'], axis=1)

In [551]:
len(kvalifikacije)

2976

In [552]:
#trke = pd.merge(kvalifikacije, trke, on=['season', 'round'], how='left')

In [553]:
rezultati = pd.read_csv('all_race_results.csv')

In [554]:
len(rezultati)

2979

In [555]:
vozaci = pd.read_csv('all_drivers.csv')

In [556]:
vozaci.columns

Index(['driverId', 'permanentNumber', 'code', 'url', 'givenName', 'familyName',
       'dateOfBirth', 'nationality'],
      dtype='object')

In [557]:
vozaci = vozaci.rename(columns={'familyName': 'driver'})
vozaci = vozaci.drop(['code', 'url', 'givenName','dateOfBirth', 
                      'nationality', 'driverId'], axis=1)

In [558]:
rezultati.columns

Index(['season', 'round', 'raceName', 'date', 'circuit', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time',
       'total_time_ms', 'points', 'fastest_lap_rank', 'fastest_lap_number',
       'fastest_lap_time', 'fastest_lap_speed', 'fastest_lap_speed_unit'],
      dtype='object')

In [559]:
rezultati = rezultati.drop(['raceName', 'date', 'circuit', 'total_time',
            'fastest_lap_speed_unit', 'fastest_lap_rank', 
            'fastest_lap_number', 'fastest_lap_time',
            'fastest_lap_speed'], axis=1)

In [560]:
len(rezultati)

2979

In [561]:
vozaci.loc[-1] = [28, 'Hartley']
vozaci.loc[-1] = [45, 'de Vries']

vozaci.loc[-1] = [47, 'Schumacher']
vozaci.loc[-1] = [21, 'de Vries']
vozaci.loc[-1] = [30, 'Lawson']
vozaci.loc[-1] = [40, 'Lawson']
vozaci.loc[-1] = [50, 'Bearman']

In [562]:
vozaci['permanentNumber'] = vozaci['permanentNumber'].astype(int)

In [563]:
len(rezultati)

2979

In [564]:
rezultati = pd.merge(rezultati, vozaci, on=['driver'], how='left')

In [565]:
len(rezultati)

2982

In [566]:
rezultati = rezultati.rename(columns={'permanentNumber': 'drivers_num'})

In [567]:
rezultati.to_csv('molimTeRadi.csv', index=False)

In [568]:
rezultati.tail()

,season,round,position_race,status,driver,constructor,grid,laps,total_time_ms,points,drivers_num
2977,2024,24,16,+1 Lap,Magnussen,Haas F1 Team,14,57,NaN,0.0,20.0
2978,2024,24,17,Engine,Lawson,RB F1 Team,12,55,NaN,0.0,30.0
2979,2024,24,18,Collision damage,Bottas,Sauber,9,30,NaN,0.0,77.0
2980,2024,24,19,Engine,Colapinto,Williams,20,26,NaN,0.0,43.0
2981,2024,24,20,Collision,Pérez,Red Bull,10,0,NaN,0.0,11.0


ovaj merge pravi problem

In [569]:
rezultati['season'] = rezultati['season'].astype(int)
rezultati['round'] = rezultati['round'].astype(int)
kvalifikacije['season'] = kvalifikacije['season'].astype(int)
kvalifikacije['round'] = kvalifikacije['round'].astype(int)
kvalifikacije['drivers_num'] = kvalifikacije['drivers_num'].astype(int)

In [570]:
print(f'rez {len(rezultati)}, kvali {len(kvalifikacije)}')

rez 2982, kvali 2976


In [571]:
# rezultati = pd.merge(kvalifikacije, rezultati, on=['season', 
#                        'round', 'drivers_num'], indicator=True)

In [572]:
rezultati = pd.merge(kvalifikacije, rezultati, on=['season', 'round', 'drivers_num'], how='outer', indicator=True)

# # Show what’s only in race results
# print(merged[merged['_merge'] == 'right_only'][['season', 'round', 'drivers_num']].drop_duplicates())


In [573]:
rezultati.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', '_merge'],
      dtype='object')

In [574]:
print(len(rezultati))

3009


In [575]:
#print(rezultati[rezultati['_merge'] == 'left_only'][['season', 'round']].drop_duplicates())

In [576]:
len(rezultati)

3009

In [577]:
rezultati = rezultati.rename(columns={'position_x': 'position_race'})
rezultati = rezultati.rename(columns={'position_y': 'position_quali'})

In [578]:
trke.head()

,season,round,race_name,circuit,lap_length,number_of_laps,number_of_corners,pit_lane_time_loss,average_overtakes_per_circuit,average_pit_stops_per_race,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend
0,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,5.303,58.0,14.0,NaN,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
1,2018,2,Bahrain Grand Prix,Bahrain International Circuit,5.412,57.0,15.0,NaN,NaN,NaN,27.982524,47.363107,1009.494175,32.198058,0.958252,0,4,False
2,2018,3,Chinese Grand Prix,Shanghai International Circuit,5.451,56.0,16.0,NaN,NaN,NaN,19.446429,24.089286,1018.131250,37.019643,1.837500,1,3,False
3,2018,4,Azerbaijan Grand Prix,Baku City Circuit,6.003,51.0,20.0,,NaN,NaN,16.661404,45.651754,1021.913158,25.251754,2.222807,0,3,False
4,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,4.655,66.0,14.0,NaN,NaN,NaN,16.050476,52.286667,1001.541905,32.339048,1.952381,1,2,False


In [579]:
rezultati['season'] = rezultati['season'].astype(int)
rezultati['round'] = rezultati['round'].astype(int)
trke['season'] = trke['season'].astype(int)
trke['round'] = trke['round'].astype(int)

In [580]:
trke = pd.merge(rezultati, trke, on=['season', 'round'], how='left')

In [581]:
len(trke)

3009

In [582]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
       'wind_direction', 'is_sprint_weekend'],
      dtype='object')

In [583]:
len(rezultati)

3009

In [584]:
#print(trke['_merge'].value_counts())

In [585]:
#missing = rezultati.merge(trke, on=['season', 'round'], how='left', indicator=True)
#print(missing[missing['_merge'] == 'left_only'][['season', 'round']])


In [586]:
trke.to_csv('trkice.csv', index=False)

In [587]:
trke.head()

,season,round,drivers_num,position_quali,position_race,status,driver,constructor,grid,laps,...,average_overtakes_per_circuit,average_pit_stops_per_race,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend
0,2018,1,2.0,12.0,9.0,Finished,Vandoorne,McLaren,11.0,58.0,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
1,2018,1,3.0,5.0,4.0,Finished,Ricciardo,Red Bull,8.0,58.0,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
2,2018,1,5.0,3.0,1.0,Finished,Vettel,Ferrari,3.0,58.0,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
3,2018,1,7.0,2.0,3.0,Finished,Räikkönen,Ferrari,2.0,58.0,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False
4,2018,1,8.0,7.0,16.0,Wheel,Grosjean,Haas F1 Team,6.0,24.0,...,NaN,NaN,24.077477,30.915315,997.003604,36.324324,3.691892,1,7,False


ako nije finished ili ne sadrzi lap onda, je driver dnf


In [588]:
trke['avg_position_last5'] = np.nan
trke['num_dnfs_last5'] = np.nan
trke['avg_gained_lost_last5'] = np.nan

In [589]:
for broj in vozaci['permanentNumber']:
    jedan_vozac = trke[trke['drivers_num'] == broj]
    
    jedan_vozac = jedan_vozac.drop(['total_time_ms',
                    'race_name', 'circuit', ' lap_length',
                    ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
                    ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
                    'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
                    'wind_direction', 'is_sprint_weekend', 'constructor'], axis=1)
    
    jedan_vozac = jedan_vozac.sort_values(by=['season', 'round'])
    for i in range(5, len(jedan_vozac)):
        last5 = jedan_vozac.iloc[i-5:i]
        current_race = jedan_vozac.iloc[i]

        # Force conversion to numeric (auto-handles strings that look like numbers)
        last5 = last5.copy()
        last5["position_race"] = pd.to_numeric(last5["position_race"], errors="coerce")
        last5["position_quali"] = pd.to_numeric(last5["position_quali"], errors="coerce")



        avg_position = last5["position_race"].astype(float).mean()
        num_dnfs = (~last5["status"].str.contains("Finished|Lap")).sum()
        gained_lost = (last5["grid"] - last5["position_race"]).astype(float)
        avg_gained_lost = gained_lost.mean()  


        idx = jedan_vozac.index[i]
        trke.loc[idx, 'avg_position_last5'] = avg_position
        trke.loc[idx, 'num_dnfs_last5'] = num_dnfs
        trke.loc[idx, 'avg_gained_lost_last5'] = avg_gained_lost

TypeError: bad operand type for unary ~: 'float'

drivers_num  position_quali season  round  position_race    status  driver  grid  laps  points

In [ ]:
trke = trke.sort_values(by=['season', 'round', 'position_race'])

In [ ]:
trke.tail()

,season,round,drivers_num,position_quali,position_race,status,driver,constructor,grid,laps,...,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction,is_sprint_weekend,avg_position_last5,num_dnfs_last5,avg_gained_lost_last5
2974,2024,24,20,15.0,16.0,+1 Lap,Magnussen,Haas F1 Team,14.0,57.0,...,51.445946,1017.426351,31.805405,1.900676,0,3,False,11.6,1.0,-1.4
2979,2024,24,30,12.0,17.0,Engine,Lawson,RB F1 Team,12.0,55.0,...,51.445946,1017.426351,31.805405,1.900676,0,3,False,12.8,0.0,0.6
2986,2024,24,77,9.0,18.0,Collision damage,Bottas,Sauber,9.0,30.0,...,51.445946,1017.426351,31.805405,1.900676,0,3,False,14.6,0.0,0.2
2981,2024,24,43,19.0,19.0,Engine,Colapinto,Williams,20.0,26.0,...,51.445946,1017.426351,31.805405,1.900676,0,3,False,14.4,2.0,-1.2
2970,2024,24,11,10.0,20.0,Collision,Pérez,Red Bull,10.0,0.0,...,51.445946,1017.426351,31.805405,1.900676,0,3,False,12.4,1.0,0.2


In [ ]:
trke.columns

Index(['season', 'round', 'drivers_num', 'position_quali', 'position_race',
       'status', 'driver', 'constructor', 'grid', 'laps', 'total_time_ms',
       'points', '_merge', 'race_name', 'circuit', ' lap_length',
       ' number_of_laps', ' number_of_corners', ' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race',
       'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindSpeed', 'rain_bin',
       'wind_direction', 'is_sprint_weekend', 'avg_position_last5',
       'num_dnfs_last5', 'avg_gained_lost_last5'],
      dtype='object')

In [ ]:
trke = trke.drop([' pit_lane_time_loss',
       ' average_overtakes_per_circuit', ' average_pit_stops_per_race'], axis=1)

In [ ]:
trke.to_csv('trke.csv', index=False)

In [ ]:
len(trke)

2988